# LeetCode 3687 — Library Late Fee Calculator bằng Python

Link đề: https://leetcode.com/problems/library-late-fee-calculator/ · Độ khó: **Easy** · Chủ đề: Array, Simulation

> **Bài Premium:** đề viết theo hiểu biết về bài (đã đối chiếu mô tả công khai), nên đối chiếu lại nếu có tài khoản Premium.

Skeleton LeetCode:

```python
class Solution:
    def lateFee(self, daysLate: List[int]) -> int:
        ...
```

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: mỗi cuốn sách trả muộn `daysLate[i]` ngày bị phạt theo **bậc**: `1` ngày phạt `1`; từ `2` đến `5` ngày phạt `2 * ngày`; hơn `5` ngày phạt `3 * ngày`; trả về **tổng tiền phạt** của mọi sách.
2. Viết hàm tính phạt cho **một** cuốn bằng `if / elif / else` và **cộng dồn** bằng `sum`.
3. Hiểu cách **chia khoảng (bậc thang)** bằng điều kiện, bằng **bảng tra** (lookup table) và bằng `bisect` trên các mốc.
4. Viết 4 cách: vòng lặp `if/elif`, bảng tra sẵn cho `1..100`, gom nhóm bằng `Counter` rồi nhân số lần, và `bisect` trên mốc; đối chiếu bằng test ngẫu nhiên có `random.seed`.
5. Tránh các lỗi: sai biên (`>= 5` thay vì `> 5`), quên ngày `1`, dùng hệ số mà không nhân với số ngày, và nhầm "mức phạt lũy tiến" với "mức phạt theo bậc".

## 2. Đề bài & ví dụ

Thư viện tính phí trả sách muộn. Cho mảng `daysLate`, trong đó `daysLate[i]` là số ngày cuốn sách thứ `i` bị trả muộn. Phí phạt của **một** cuốn:

- trả muộn đúng `1` ngày: phạt `1`;
- trả muộn từ `2` đến `5` ngày: phạt `2 * daysLate[i]`;
- trả muộn **hơn** `5` ngày: phạt `3 * daysLate[i]`.

Trả về **tổng** phí phạt của tất cả các cuốn sách.

| Ví dụ | daysLate | Output | Giải thích |
|---|---|---|---|
| 1 | `[5,1,7]` | `32` | `5` ngày: `2*5 = 10`; `1` ngày: `1`; `7` ngày: `3*7 = 21`; tổng `32` |
| 2 | `[1,1]` | `2` | mỗi cuốn phạt `1`; tổng `2` |

Constraints: `1 <= daysLate.length <= 100`, `1 <= daysLate[i] <= 100`.

In [1]:
days = [5, 1, 7]
total = 0
for d in days:
    if d == 1:
        fee = 1
    elif d <= 5:
        fee = 2 * d
    else:
        fee = 3 * d
    print(f"{d} ngày -> phạt {fee}")
    total += fee
print("tổng =", total)

5 ngày -> phạt 10
1 ngày -> phạt 1
7 ngày -> phạt 21
tổng = 32


## 3. Kiến thức nền

### 3.1. Điều kiện nhiều nhánh `if / elif / else`

Python kiểm tra các nhánh **từ trên xuống**, chạy nhánh **đầu tiên** đúng rồi bỏ qua phần còn lại. Vì vậy thứ tự nhánh quan trọng: kiểm tra `d == 1` trước, rồi `d <= 5` (lúc này đã biết `d >= 2`), cuối cùng `else` (tức `d > 5`).

In [2]:
def band(d):
    if d == 1:
        return "bậc 1"
    elif d <= 5:
        return "bậc 2..5"
    else:
        return "bậc > 5"

print([band(d) for d in (1, 2, 5, 6, 100)])

['bậc 1', 'bậc 2..5', 'bậc 2..5', 'bậc > 5', 'bậc > 5']


### 3.2. `sum` với biểu thức sinh

`sum(f(d) for d in days)` tính `f` cho từng phần tử rồi cộng lại, không cần biến cộng dồn thủ công. Với list rỗng `sum` trả `0` (đề đảm bảo không rỗng).

In [3]:
def fee(d):
    return 1 if d == 1 else 2 * d if d <= 5 else 3 * d

print(sum(fee(d) for d in [5, 1, 7]))
print(sum([]), sum([2, 3]))

32
0 5


### 3.3. Bảng tra (lookup table)

Vì `daysLate[i] <= 100`, ta có thể tính trước phí cho mọi `d` trong `0..100` vào một danh sách `FEE`, rồi tra `FEE[d]` mỗi lần (`O(1)`). Tính trước trả giá một lần, đổi lại mỗi lần tra rất nhanh và mã chính đơn giản.

In [4]:
FEE = [0] * 101
for d in range(1, 101):
    FEE[d] = 1 if d == 1 else 2 * d if d <= 5 else 3 * d
print(FEE[:9])
print(FEE[100], sum(FEE[d] for d in [5, 1, 7]))

[0, 1, 4, 6, 8, 10, 18, 21, 24]
300 32


### 3.4. `Counter` và `bisect`

- `Counter(days)` gom các ngày giống nhau: tính phạt **một lần** cho mỗi giá trị khác nhau rồi **nhân số lần**.
- `bisect.bisect_left(mốc, x)` trả vị trí chèn `x` vào danh sách đã sắp: dùng để chọn "bậc" từ một danh sách mốc, thay cho chuỗi `if/elif`.

In [5]:
from collections import Counter
from bisect import bisect_left

print(Counter([5, 1, 7, 1]))
cuts = [1, 5]                               # d <= 1 -> bậc 0; d <= 5 -> bậc 1; còn lại bậc 2
print([bisect_left(cuts, d) for d in (1, 2, 5, 6, 100)])

Counter({1: 2, 5: 1, 7: 1})
[0, 1, 1, 2, 2]


## 4. Ý tưởng nền tảng / cách nghĩ trực giác

Ví dụ 1: `[5, 1, 7]`.

```
5 ngày  -> thuộc khoảng 2..5  -> 2 * 5 = 10
1 ngày  -> đúng 1 ngày        -> 1
7 ngày  -> hơn 5 ngày         -> 3 * 7 = 21
tổng    = 10 + 1 + 21 = 32
```

Mỗi cuốn sách được tính **độc lập** rồi cộng lại: bài thuộc loại "mô phỏng theo quy tắc". Cần đọc đúng biên: `5` thuộc nhóm `2..5` (hệ số `2`), còn `6` mới thuộc nhóm hệ số `3`.

## 5. Cách 1 — Brute Force / baseline

Duyệt từng cuốn sách, dùng `if / elif / else` tính phạt, cộng vào `total`.

Nút thắt: không có — bài chỉ cần một lượt `O(n)`. Chú ý duy nhất là đọc đúng biên các bậc.

In [6]:
class SolutionLoop:
    def lateFee(self, daysLate):
        total = 0
        for d in daysLate:
            if d == 1:
                total += 1
            elif d <= 5:
                total += 2 * d
            else:
                total += 3 * d
        return total

print(SolutionLoop().lateFee([5, 1, 7]))
print(SolutionLoop().lateFee([1, 1]))

32
2


**Độ phức tạp:** Time `O(n)`, Space `O(1)`.

## 6. Ý tưởng thuật toán / bước đột phá

**Bước đột phá:** tách bài thành **hai việc nhỏ**: (a) hàm `fee(d)` cho một cuốn; (b) cộng `fee` trên cả mảng. Hàm `fee` là hàm **từng khúc** (piecewise): ba khúc theo `d`.

**Quy tắc "nếu… thì…":**

- Nếu `d == 1` -> `1`.
- Ngược lại nếu `d <= 5` -> `2 * d`.
- Ngược lại (`d > 5`) -> `3 * d`.

Vì `1 = 1 * 1`, ta có thể xem ba khúc là hệ số `1, 2, 3` nhân với `d`, với hệ số `1` chỉ áp dụng cho `d = 1`. Kết quả cuối là `sum(fee(d))`.

In [7]:
def fee(d):
    if d == 1:
        return 1
    if d <= 5:
        return 2 * d
    return 3 * d

print([(d, fee(d)) for d in (1, 2, 3, 4, 5, 6, 7, 100)])
print(sum(fee(d) for d in [5, 1, 7]))

[(1, 1), (2, 4), (3, 6), (4, 8), (5, 10), (6, 18), (7, 21), (100, 300)]
32


## 7. Các cách tiếp theo

Tổng cộng 4 cách: (1) vòng lặp `if/elif`; (2) bảng tra sẵn `FEE[0..100]`; (3) gom nhóm bằng `Counter` rồi nhân số lần; (4) `bisect` trên các mốc để chọn hệ số.

### Cách 2 — Bảng tra sẵn

Tính `FEE[d]` một lần cho `d = 1..100`; mỗi lần gọi chỉ cần `sum(FEE[d] for d in daysLate)`.

In [8]:
class SolutionTable:
    FEE = None

    def __init__(self):
        if SolutionTable.FEE is None:
            table = [0] * 101
            for d in range(1, 101):
                table[d] = 1 if d == 1 else (2 * d if d <= 5 else 3 * d)
            SolutionTable.FEE = table

    def lateFee(self, daysLate):
        return sum(self.FEE[d] for d in daysLate)

print(SolutionTable().lateFee([5, 1, 7]))
print(SolutionTable().lateFee([1, 1]))

32
2


Độ phức tạp Cách 2: Time `O(n)` (+ `O(100)` dựng bảng một lần), Space `O(101)`.

### Cách 3 — Gom nhóm bằng `Counter`

Đếm mỗi giá trị `d` xuất hiện `c` lần; cuốn có `d` ngày đóng góp `c * fee(d)`. Hữu ích khi mảng rất dài nhưng ít giá trị khác nhau.

In [9]:
from collections import Counter

class SolutionCounter:
    @staticmethod
    def _fee(d):
        if d == 1:
            return 1
        return 2 * d if d <= 5 else 3 * d

    def lateFee(self, daysLate):
        total = 0
        for d, c in Counter(daysLate).items():
            total += c * self._fee(d)
        return total

print(SolutionCounter().lateFee([5, 1, 7]))
print(SolutionCounter().lateFee([1, 1]))
print(SolutionCounter().lateFee([7, 7, 7, 1, 1]))

32
2
65


Độ phức tạp Cách 3: Time `O(n)` (đếm) + `O(k)` với `k` giá trị khác nhau, Space `O(k)`.

### Cách 4 — `bisect` chọn hệ số

Với mỗi `d`, ta tra "bậc" bằng `bisect_left([1, 5], d)`: `d = 1` -> `0`, `2..5` -> `1`, `>= 6` -> `2`. Hệ số tương ứng `[1, 2, 3]`. Phí `= hệ số * d`. Cách này gọn khi có **nhiều bậc**.

In [10]:
from bisect import bisect_left

class SolutionBisect:
    CUTS = [1, 5]
    RATE = [1, 2, 3]

    def lateFee(self, daysLate):
        total = 0
        for d in daysLate:
            total += self.RATE[bisect_left(self.CUTS, d)] * d
        return total

print(SolutionBisect().lateFee([5, 1, 7]))
print(SolutionBisect().lateFee([1, 1]))
print([bisect_left([1, 5], d) for d in (1, 2, 5, 6)])

32
2
[0, 1, 1, 2]


Độ phức tạp Cách 4: Time `O(n log b)` với `b` = số mốc (ở đây `b = 2`, coi như `O(n)`), Space `O(1)`.

## 8. TRACE từng bước

Trace Cách 1 với `daysLate = [5, 1, 7]`:

| bước | `d` | nhánh | phí | `total` sau |
|---|---|---|---|---|
| 1 | 5 | `d <= 5` | `2*5 = 10` | 10 |
| 2 | 1 | `d == 1` | 1 | 11 |
| 3 | 7 | `else` | `3*7 = 21` | 32 |

In [11]:
def trace_fee(daysLate):
    total = 0
    for step, d in enumerate(daysLate, start=1):
        if d == 1:
            branch, f = "d == 1", 1
        elif d <= 5:
            branch, f = "d <= 5", 2 * d
        else:
            branch, f = "else (d > 5)", 3 * d
        total += f
        print(f"bước {step}: d={d} | nhánh {branch} | phí={f} | total={total}")
    return total

trace_fee([5, 1, 7])
trace_fee([6, 2, 100])

bước 1: d=5 | nhánh d <= 5 | phí=10 | total=10
bước 2: d=1 | nhánh d == 1 | phí=1 | total=11
bước 3: d=7 | nhánh else (d > 5) | phí=21 | total=32
bước 1: d=6 | nhánh else (d > 5) | phí=18 | total=18
bước 2: d=2 | nhánh d <= 5 | phí=4 | total=22
bước 3: d=100 | nhánh else (d > 5) | phí=300 | total=322


322

## 9. Code nộp LeetCode

In [12]:
from typing import List

class Solution:
    def lateFee(self, daysLate: List[int]) -> int:
        total = 0
        for d in daysLate:
            if d == 1:
                total += 1
            elif d <= 5:
                total += 2 * d
            else:
                total += 3 * d
        return total

print(Solution().lateFee([5, 1, 7]), Solution().lateFee([1, 1]))

32 2


## 10. Vì sao thuật toán đúng

- **Phân hoạch đầy đủ:** với số nguyên `d >= 1`, ba trường hợp `d == 1`, `2 <= d <= 5`, `d > 5` bao phủ mọi `d` và **không giao nhau**; thứ tự `if / elif / else` đảm bảo đúng nhánh.
- **Tính cộng tính:** tổng phí là tổng phí từng cuốn (đề không có chiết khấu hay phụ thu theo số sách), nên `sum(fee(d))` đúng.
- **Cách 2:** `FEE[d]` được tính bằng đúng công thức của `fee`, nên giống từng phần tử.
- **Cách 3:** `Counter` gom `c` cuốn cùng `d`; tổng phí của nhóm là `c * fee(d)` (cộng `c` lần cùng một số).
- **Cách 4:** `bisect_left([1, 5], d)` cho `0` khi `d <= 1`, `1` khi `1 < d <= 5`, `2` khi `d > 5` — trùng ba khúc; `d = 1` có hệ số `1`, đúng vì `fee(1) = 1 = 1*1`.
- **Biên quan trọng:** `d = 5` thuộc khúc hệ số `2` (phạt `10`), `d = 6` thuộc khúc hệ số `3` (phạt `18`); phí **nhảy** từ `10` lên `18` chứ không tăng đều.

## 11. Phân tích Time / Space Complexity

`n = len(daysLate) <= 100`, giá trị `d <= 100`, `k` số giá trị khác nhau.

| Cách | Time | Space |
|---|---|---|
| 1. Vòng lặp `if/elif` | `O(n)` | `O(1)` |
| 2. Bảng tra sẵn | `O(n)` (+ dựng bảng `O(100)`) | `O(101)` |
| 3. `Counter` + nhân số lần | `O(n + k)` | `O(k)` |
| 4. `bisect` chọn hệ số | `O(n log b)`, `b = 2` | `O(1)` |

Mọi cách đều tức thì; Cách 1 đơn giản nhất và là cách nộp.

## 12. Bộ test

Test 1: ví dụ đề và edge case (đều tuân `1 <= len <= 100`, `1 <= d <= 100`). Các biên `1, 2, 5, 6` được kiểm tra riêng.

In [13]:
impls = [SolutionLoop(), SolutionTable(), SolutionCounter(), SolutionBisect(), Solution()]

def oracle(days):
    # mô phỏng độc lập: tính từng cuốn theo định nghĩa
    total = 0
    for d in days:
        if d == 1:
            total += 1
        if 2 <= d <= 5:
            total += 2 * d
        if d > 5:
            total += 3 * d
    return total

tests = [
    ([5, 1, 7], 32),
    ([1, 1], 2),
    ([1], 1),
    ([2], 4),                  # biên dưới nhóm 2..5
    ([5], 10),                 # biên trên nhóm 2..5
    ([6], 18),                 # biên dưới nhóm > 5
    ([100], 300),              # lớn nhất
    ([3, 4], 14),
    ([1, 2, 5, 6], 1 + 4 + 10 + 18),
    ([100] * 100, 30000),      # độ dài tối đa
    ([1] * 100, 100),
]
for days, exp in tests:
    assert 1 <= len(days) <= 100 and all(1 <= d <= 100 for d in days)
    assert oracle(days) == exp, (days, oracle(days), exp)
    for im in impls:
        assert im.lateFee(list(days)) == exp, (type(im).__name__, days)
print("tất cả test cố định OK:", len(tests))

tất cả test cố định OK: 11


Test 2: ngẫu nhiên có seed, đúng constraints.

In [14]:
import random
random.seed(3687)

for t in range(4000):
    n = random.randint(1, 100)
    hi = random.choice([1, 2, 5, 6, 10, 100])
    days = [random.randint(1, hi) for _ in range(n)]
    exp = oracle(days)
    for im in impls:
        assert im.lateFee(list(days)) == exp, (type(im).__name__, days)
print("4000 test ngẫu nhiên OK")

4000 test ngẫu nhiên OK


Test 3: tính chất — phí là **cộng tính** (phí ghép hai danh sách = tổng hai phí); không phụ thuộc thứ tự; phí mỗi cuốn `>= d` (hệ số `>= 1`) nên tổng `>= sum(days)`; và mỗi cuốn tối đa `3 * d`.

In [15]:
random.seed(687)
for _ in range(1500):
    a = [random.randint(1, 100) for _ in range(random.randint(1, 50))]
    b = [random.randint(1, 100) for _ in range(random.randint(1, 50))]
    fa, fb = Solution().lateFee(a), Solution().lateFee(b)
    assert Solution().lateFee(a + b) == fa + fb
    c = a[:]
    random.shuffle(c)
    assert Solution().lateFee(c) == fa
    assert sum(a) <= fa <= 3 * sum(a)
print("tính chất (cộng tính, hoán vị, cận dưới sum, cận trên 3*sum): OK")

tính chất (cộng tính, hoán vị, cận dưới sum, cận trên 3*sum): OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1: sai biên (`>= 5` thay vì `> 5`)

In [16]:
def wrong_boundary(days):
    total = 0
    for d in days:
        if d == 1:
            total += 1
        elif d < 5:                              # SAI: d = 5 rơi vào nhóm hệ số 3
            total += 2 * d
        else:
            total += 3 * d
    return total

print("sai :", wrong_boundary([5, 1, 7]))
print("đúng:", Solution().lateFee([5, 1, 7]))
assert wrong_boundary([5, 1, 7]) != Solution().lateFee([5, 1, 7])

sai : 37
đúng: 32


Vì sao sai: `5` ngày vẫn thuộc nhóm `2..5` (hệ số `2`, phạt `10`), nhưng điều kiện `d < 5` đưa nó sang hệ số `3` (phạt `15`) nên tổng thành `37`. **Sửa:** `d <= 5`.

### Lỗi 2: quên trường hợp ngày `1` (dùng `2 * d` cho `d <= 5`)

In [17]:
def wrong_no_one(days):
    total = 0
    for d in days:
        if d <= 5:
            total += 2 * d                       # SAI: d = 1 phải phạt 1, không phải 2
        else:
            total += 3 * d
    return total

print("sai :", wrong_no_one([1, 1]))
print("đúng:", Solution().lateFee([1, 1]))
assert wrong_no_one([1, 1]) != Solution().lateFee([1, 1])

sai : 4
đúng: 2


Vì sao sai: đề riêng cho ngày `1` mức phạt `1`; công thức `2 * d` cho `2`. **Sửa:** nhánh `d == 1` đặt **trước**.

### Lỗi 3: dùng hệ số mà không nhân với số ngày

In [18]:
def wrong_flat(days):
    total = 0
    for d in days:
        if d == 1:
            total += 1
        elif d <= 5:
            total += 2                           # SAI: thiếu * d
        else:
            total += 3                           # SAI: thiếu * d
    return total

print("sai :", wrong_flat([5, 1, 7]))
print("đúng:", Solution().lateFee([5, 1, 7]))
assert wrong_flat([5, 1, 7]) != Solution().lateFee([5, 1, 7])

sai : 6
đúng: 32


Vì sao sai: phí tỉ lệ với **số ngày** (`2 * daysLate[i]`), không phải hằng số theo bậc. **Sửa:** nhân `d`.

### Lỗi 4: nhầm với phí lũy tiến (tính từng khúc như thuế)

In [19]:
def wrong_marginal(days):
    total = 0
    for d in days:
        # SAI: tính "ngày đầu 1, 4 ngày kế 2, phần còn lại 3" (lũy tiến)
        total += 1 + 2 * (min(d, 5) - 1) + 3 * max(d - 5, 0)
    return total

print("sai :", wrong_marginal([5, 1, 7]))
print("đúng:", Solution().lateFee([5, 1, 7]))
assert wrong_marginal([5, 1, 7]) != Solution().lateFee([5, 1, 7])

sai : 25
đúng: 32


Vì sao sai: đề không dùng lũy tiến; **toàn bộ** số ngày được nhân cùng một hệ số theo nhóm của nó (`7` ngày -> `3 * 7 = 21`, chứ không phải `1 + 2*4 + 3*2`).

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Mỗi phần tử tính độc lập theo luật, rồi cộng" -> hàm `fee(x)` + `sum`.
- Luật chia khoảng -> `if / elif / else` từ **hẹp đến rộng**; viết rõ biên bằng ví dụ ở biên (`1, 2, 5, 6`).
- Miền nhỏ (`<= 100`) -> có thể **bảng tra**; nhiều bậc -> `bisect` trên danh sách mốc.
- Phân biệt: "hệ số theo bậc áp cho **toàn bộ**" và "lũy tiến áp cho **từng phần**".
- Nên viết thêm oracle bằng **các `if` độc lập** (không dùng `elif`) để kiểm tra chéo.
- Khi `n` lớn nhưng ít giá trị khác nhau -> `Counter`.

## 15. Biến thể của bài

1. Mức phạt **lũy tiến** (như thuế): ngày đầu một giá, các ngày sau giá khác.
2. Thêm **phí tối đa** mỗi cuốn (`min(phí, trần)`).
3. Giảm giá nếu **tổng số sách trả muộn** vượt một ngưỡng.
4. Trả về **số cuốn** thuộc từng nhóm bậc (dùng `Counter` của `bisect`).
5. Tính phí khi biết **ngày hẹn trả** và **ngày trả thực tế** (hiệu hai ngày, dùng `datetime`).

## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Vòng lặp `if/elif` | tính từng cuốn | `O(n)` | `O(1)` | **nên nộp**, rõ nhất |
| 2. Bảng tra sẵn | `FEE[d]` | `O(n)` | `O(101)` | nhiều truy vấn, miền nhỏ |
| 3. `Counter` + nhân | gom ngày giống nhau | `O(n + k)` | `O(k)` | ít giá trị khác nhau |
| 4. `bisect` hệ số | chọn bậc bằng mốc | `O(n log b)` | `O(1)` | nhiều bậc, dữ liệu cấu hình |

## 17. 5 bài tập vận dụng

**Bài 1 (đoán output):** `Solution().lateFee([2, 6])` là gì?

**Bài 2 (tính tay):** `daysLate = [1, 3, 5, 6, 10]`. Phí từng cuốn và tổng?

**Bài 3 (điền TODO):** hoàn thành `fee_capped(d, cap)` — phí một cuốn nhưng không vượt `cap`.

**Bài 4 (tự viết lại):** viết `count_bands(daysLate)` trả về dict `{1: số cuốn ngày 1, 2: số cuốn nhóm 2..5, 3: số cuốn nhóm > 5}`.

**Bài 5 (mini challenge):** viết `fee_progressive(daysLate)` tính phí **lũy tiến**: ngày đầu `1`, ngày `2..5` mỗi ngày `2`, từ ngày `6` mỗi ngày `3`; so sánh với `Solution().lateFee` và tìm một ví dụ khác nhau.

In [20]:
# Bài 1: đoán output rồi chạy để kiểm tra
print(Solution().lateFee([2, 6]))

# Bài 3: TODO (KHÔNG gọi hàm này khi chạy notebook)
def fee_capped(d, cap):
    # TODO: tính fee(d) theo ba nhóm rồi trả min(fee, cap)
    return -1

22


## 18. Gợi ý 3 mức cho bài tập

**Mức 1 (nhẹ):** xác định nhóm của từng số (`1`, `2..5`, `> 5`) rồi nhân hệ số.

**Mức 2 (vừa):** Bài 3: dùng hàm `fee` ở mục 6 rồi `min`. Bài 4: duyệt và tăng bộ đếm nhóm. Bài 5: `1 + 2 * (min(d, 5) - 1) + 3 * max(d - 5, 0)` cho mỗi cuốn.

**Mức 3 (gần đáp án):** Bài 1: `2 -> 4`, `6 -> 18` -> `22`. Bài 2: `1, 6, 10, 18, 30` -> tổng `65`.

## 19. Đáp án tham khảo cho bài tập

In [21]:
# Bài 1
assert Solution().lateFee([2, 6]) == 22

# Bài 2
assert [fee(d) for d in [1, 3, 5, 6, 10]] == [1, 6, 10, 18, 30]
assert Solution().lateFee([1, 3, 5, 6, 10]) == 65

# Bài 3
def fee_capped(d, cap):
    return min(fee(d), cap)
assert fee_capped(100, 50) == 50
assert fee_capped(2, 50) == 4
assert fee_capped(1, 0) == 0

# Bài 4
def count_bands(daysLate):
    res = {1: 0, 2: 0, 3: 0}
    for d in daysLate:
        if d == 1:
            res[1] += 1
        elif d <= 5:
            res[2] += 1
        else:
            res[3] += 1
    return res
assert count_bands([5, 1, 7]) == {1: 1, 2: 1, 3: 1}
assert count_bands([1, 1, 2, 3, 9]) == {1: 2, 2: 2, 3: 1}

# Bài 5
def fee_progressive(daysLate):
    return sum(1 + 2 * (min(d, 5) - 1) + 3 * max(d - 5, 0) for d in daysLate)
assert fee_progressive([1, 1]) == Solution().lateFee([1, 1])
assert fee_progressive([5, 1, 7]) == 9 + 1 + 15
assert fee_progressive([5, 1, 7]) != Solution().lateFee([5, 1, 7])
print("đáp án bài tập OK")

đáp án bài tập OK


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template "tính theo bậc cho từng phần tử rồi cộng":**

```python
total = 0
for d in daysLate:
    if d == 1:
        total += 1
    elif d <= 5:
        total += 2 * d
    else:
        total += 3 * d
return total
```

**Checklist trước khi submit:**

- [ ] Nhánh `d == 1` nằm **trước** nhánh `d <= 5`?
- [ ] Biên đúng: `5` thuộc hệ số `2`, `6` thuộc hệ số `3` (`d <= 5`, không phải `d < 5`)?
- [ ] Nhân **hệ số với số ngày** (không cộng hằng số)?
- [ ] Không tính lũy tiến; toàn bộ số ngày dùng chung một hệ số?
- [ ] Đã thử `[1]`, `[2]`, `[5]`, `[6]`, `[100]` và ví dụ đề (`32`, `2`)?
- [ ] Chữ ký đúng `lateFee(self, daysLate: List[int]) -> int`?